# Air Quality Europe  ETL Pipeline
**Proyecto TFM MD008/MD009 La Salle, Ramon Llull**

Pipeline completo de extracción y transformación de datos de calidad del aire en Europa usando OpenAQ API v3 y AWS S3.

**Tablas generadas:**
- `Paises.parquet`  países europeos
- `Locations.parquet` estaciones de monitoreo
- `Sensores.parquet` sensores por estación
- `measurements_2023_MM.parquet` mediciones por mes (fact table)

# Air Quality Europe — ETL Pipeline
**Proyecto TFM — MD008/MD009 — La Salle, Ramon Llull**

Pipeline completo de extracción y transformación de datos de calidad del aire en Europa usando OpenAQ API v3 y AWS S3.

**Tablas generadas:**
- `Paises.parquet` — países europeos
- `Locations.parquet` — estaciones de monitoreo
- `Sensores.parquet` — sensores por estación
- `measurements_2023_MM.parquet` — mediciones por mes (fact table)

## 0. Instalación de dependencias

In [2]:
# Dependencias ya instaladas en el contenedor Docker (pyspark, boto3)
print('Dependencias listas')

Dependencias listas


## 1. Imports y configuración general

In [3]:
import requests
import json
import time
import os
import boto3
import pandas as pd
from botocore import UNSIGNED
from botocore.config import Config
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

# API OpenAQ
apiKey = 'c6d9704084e955dfd77c6a1a68ace0ed067345807a61d3f1405d0e2f3e427658'
url = 'https://api.openaq.org/v3'
headers = {'X-API-Key': apiKey}

# Cerrar sesión anterior si existe
if 'spark' in locals():
    spark.stop()

spark = SparkSession.builder \
    .appName('OpenAQ_ETL') \
    .master('local[*]') \
    .config('spark.driver.memory', '8g') \
    .config('spark.sql.shuffle.partitions', '16') \
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 
            'org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider') \
    .getOrCreate()

print('Versión de Spark:', spark.version)

# Carpeta destino — en Docker usamos el volumen montado en /app/data
# (en Colab era /content/drive/MyDrive/openaq_2023)
output_dir = '/app/data'
os.makedirs(output_dir, exist_ok=True)
print(f'Carpeta destino: {output_dir}')

/usr/local/lib/python3.11/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 08:18:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Versión de Spark: 3.5.1
Carpeta destino: /app/data


## 2. Tabla de Países

Obtenemos todos los países de OpenAQ y filtramos los europeos por código ISO.

In [3]:
response = requests.get(f'{url}/countries', headers=headers, params={'limit': 500})
aqResponse = response.json()
print(f'Total países en OpenAQ: {aqResponse["meta"]["found"]}')
print(f'Status code: {response.status_code}')

datosPaises = aqResponse['results']

# Códigos ISO de países europeos
euCodes = [
    'AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR',
    'DE', 'GR', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL',
    'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE',
    'NO', 'CH', 'GB', 'IS', 'RS', 'BA', 'ME', 'MK', 'AL'
]

euPaises = []
for row in datosPaises:
    if row['code'] in euCodes:
        euPaises.append({
            'ID_Paises': row['id'],
            'Pais_Code': row['code'],
            'Pais': row['name']
        })

Paises = pd.DataFrame(euPaises)
Paises.to_parquet(f'{output_dir}/Paises.parquet', index=False)
print(f'N° de países europeos: {len(euPaises)}')
Paises.head()

Total países en OpenAQ: 150
Status code: 200
N° de países europeos: 36


,ID_Paises,Pais_Code,Pais
0,8,CY,Cyprus
1,22,FR,France
2,44,LT,Lithuania
3,49,CZ,Czech Republic
4,50,DE,Germany


## 3. Tabla de Locations

Descargamos todas las estaciones europeas, paginando por país.

In [4]:
euLocations = []

for pais in euPaises:
    page = 1
    while True:
        response = requests.get(f'{url}/locations', headers=headers,
            params={'limit': 1000, 'countries_id': pais['ID_Paises'], 'page': page})
        data = response.json()

        if 'results' not in data:
            print(f' Sin resultados para {pais["Pais_Code"]} (page {page}): {data}')
            break

        if not data['results']:
            break

        euLocations.extend(data['results'])

        if len(data['results']) < 1000:
            break

        page += 1
        time.sleep(0.3)

    print(f'  {pais["Pais_Code"]}: descargado — total acumulado: {len(euLocations)}')

print(f'\nTotal locations descargadas: {len(euLocations)}')

  CY: descargado — total acumulado: 15
  FR: descargado — total acumulado: 988
  LT: descargado — total acumulado: 1010
  CZ: descargado — total acumulado: 1163
  DE: descargado — total acumulado: 1909
  EE: descargado — total acumulado: 1920
  LV: descargado — total acumulado: 1939
  NO: descargado — total acumulado: 2034
  SE: descargado — total acumulado: 2235
  FI: descargado — total acumulado: 2321
  LU: descargado — total acumulado: 2335
  BE: descargado — total acumulado: 2605
  MK: descargado — total acumulado: 2639
  AL: descargado — total acumulado: 2646
  ES: descargado — total acumulado: 3559


26/04/06 12:12:06 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


  DK: descargado — total acumulado: 3591
  RO: descargado — total acumulado: 3996
  HU: descargado — total acumulado: 4116
  SK: descargado — total acumulado: 4207
  PL: descargado — total acumulado: 4637
  IE: descargado — total acumulado: 4739
  GB: descargado — total acumulado: 5435
  GR: descargado — total acumulado: 5509
  AT: descargado — total acumulado: 5939
  IT: descargado — total acumulado: 6752
  CH: descargado — total acumulado: 6844
  NL: descargado — total acumulado: 7060
  RS: descargado — total acumulado: 7177
  HR: descargado — total acumulado: 7225
  SI: descargado — total acumulado: 7287
  BG: descargado — total acumulado: 7419
  ME: descargado — total acumulado: 7434
  BA: descargado — total acumulado: 7486
  PT: descargado — total acumulado: 7575
  IS: descargado — total acumulado: 7610
  MT: descargado — total acumulado: 7620

Total locations descargadas: 7620


In [5]:
# Aplanar el JSON anidado en un DataFrame limpio
euLocationsFlat = []

for loc in euLocations:
    euLocationsFlat.append({
        'location_id':    loc['id'],
        'location_name':  loc['name'],
        'country_id':     loc['country']['id'],
        'country_code':   loc['country']['code'],
        'provider_id':    loc['provider']['id'],
        'provider_name':  loc['provider']['name'],
        'is_monitor':     loc['isMonitor'],
        'lat':            loc['coordinates']['latitude'],
        'lon':            loc['coordinates']['longitude'],
        'datetime_first': (loc['datetimeFirst'] or {}).get('utc'),
        'datetime_last':  (loc['datetimeLast'] or {}).get('utc')
    })

Locations = pd.DataFrame(euLocationsFlat)
Locations.to_parquet(f'{output_dir}/Locations.parquet', index=False)
print(f'Locations guardadas: {len(Locations)}')
Locations.head()

Locations guardadas: 7620


,location_id,location_name,country_id,country_code,provider_id,provider_name,is_monitor,lat,lon,datetime_first,datetime_last
0,8161,"""NICOSIA RESIDENTIAL""",8,CY,70,EEA,True,35.126942,33.331665,2020-04-20T16:00:00Z,2026-04-06T11:00:00Z
1,663493,Limassol - Traffic Station,8,CY,52,Republic of Cyprus Department of Labor Inspection,True,34.686111,33.035556,2023-03-19T01:00:00Z,2026-04-06T11:00:00Z
2,663494,Paralimni - Traffic Station,8,CY,52,Republic of Cyprus Department of Labor Inspection,True,35.045693,33.977735,2023-03-19T01:00:00Z,2026-04-06T11:00:00Z
3,663495,Kalavasos - Industrial Station,8,CY,52,Republic of Cyprus Department of Labor Inspection,True,34.771530,33.296230,2023-03-19T01:00:00Z,2026-04-06T11:00:00Z
4,663496,Mari - Industrial Station,8,CY,52,Republic of Cyprus Department of Labor Inspection,True,34.740154,33.300276,2023-03-19T01:00:00Z,2026-04-06T11:00:00Z


## 4. Tabla de Sensores

Extraemos los sensores de cada location, filtrando por parámetros de interés.

In [6]:
PARAMS_INTERES = {'pm25', 'pm10', 'no2', 'o3', 'so2', 'co'}

sensors = []
for loc in euLocations:
    for sens in loc['sensors']:
        param = sens['parameter']['name']
        if param in PARAMS_INTERES:
            sensors.append({
                'sensor_id':   sens['id'],
                'location_id': loc['id'],
                'param':       param,
                'units':       sens['parameter']['units']
            })

Sensores = pd.DataFrame(sensors)
Sensores.to_parquet(f'{output_dir}/Sensores.parquet', index=False)
print(f'Sensores guardados: {len(Sensores)}')
Sensores.head()

Sensores guardados: 22072


,sensor_id,location_id,param,units
0,23770,8161,co,µg/m³
1,23757,8161,no2,µg/m³
2,23754,8161,o3,µg/m³
3,23755,8161,so2,µg/m³
4,3646860,663493,co,µg/m³


## 5. Tabla de Measurements

Descargamos las mediciones del año 2023 desde el bucket público de OpenAQ en S3.

**Estrategia:**
1. boto3 verifica qué locations tienen datos en S3 (una llamada por location)
2. Para cada mes, Spark lee todos los CSV en paralelo
3. Guardamos un parquet por mes en /app/data (volumen montado local)

In [7]:
# Conexión anónima a S3
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# Solo locations con isMonitor=True, locations oficiales
eu_location_ids = Locations[Locations['is_monitor'] == True]['location_id'].tolist()
print(f'Total locations a verificar: {len(eu_location_ids)}')

Total locations a verificar: 6177


In [8]:
# Verificación optimizada: una sola llamada a S3 por location para todo 2023
# En lugar de 12 llamadas (una por mes), listamos todo el año de una vez

paths_por_mes = {f'{m:02d}': [] for m in range(1, 13)}

for i, loc_id in enumerate(eu_location_ids):

    prefix = f'records/csv.gz/locationid={loc_id}/year=2023/'
    response = s3.list_objects_v2(Bucket='openaq-data-archive', Prefix=prefix)

    if response.get('KeyCount', 0) == 0:
        continue

    # Extraer qué meses tiene esta location del listado de archivos
    meses_encontrados = set()
    for obj in response.get('Contents', []):
        key = obj['Key']
        mes = key.split('month=')[1].split('/')[0]
        meses_encontrados.add(mes)

    for mes in meses_encontrados:
        paths_por_mes[mes].append(
            f's3a://openaq-data-archive/records/csv.gz/locationid={loc_id}/year=2023/month={mes}/'
        )

    if (i + 1) % 500 == 0:
        print(f'  Procesadas {i + 1} de {len(eu_location_ids)} locations...')

print('\nVerificación completa')
for mes, paths in paths_por_mes.items():
    print(f'  Mes {mes}: {len(paths)} paths válidos')

  Procesadas 500 de 6177 locations...
  Procesadas 1000 de 6177 locations...
  Procesadas 2500 de 6177 locations...
  Procesadas 3500 de 6177 locations...
  Procesadas 4000 de 6177 locations...
  Procesadas 5000 de 6177 locations...

Verificación completa
  Mes 01: 3139 paths válidos
  Mes 02: 2637 paths válidos
  Mes 03: 2432 paths válidos
  Mes 04: 3106 paths válidos
  Mes 05: 3256 paths válidos
  Mes 06: 3233 paths válidos
  Mes 07: 3246 paths válidos
  Mes 08: 2981 paths válidos
  Mes 09: 3130 paths válidos
  Mes 10: 3213 paths válidos
  Mes 11: 3251 paths válidos
  Mes 12: 3259 paths válidos


In [9]:
from datetime import datetime

meses = [f'{m:02d}' for m in range(1, 13)]

for mes in meses:

    output_path = f'{output_dir}/measurements_2023_{mes}.parquet'

    if os.path.exists(output_path):
        print(f'Mes {mes} ya procesado, saltando...')
        continue

    inicio = datetime.now()
    print(f'\nProcesando mes: {mes} — inicio: {inicio.strftime("%H:%M:%S")}')

    paths_validos = paths_por_mes[mes]

    if not paths_validos:
        print(f'Sin datos para mes {mes}, saltando...')
        continue

    schema = StructType([
        StructField("location_id", IntegerType()),
        StructField("sensors_id",  IntegerType()),
        StructField("location",    StringType()),
        StructField("datetime",    StringType()),
        StructField("lat",         DoubleType()),
        StructField("lon",         DoubleType()),
        StructField("parameter",   StringType()),
        StructField("units",       StringType()),
        StructField("value",       DoubleType())
    ])

    # Leer directo desde S3 sin descargar a /tmp/
    df_mes = spark.read \
        .option('header', 'true') \
        .schema(schema) \
        .csv(paths_validos)

    # --- TRANSFORMACIONES ---
    df_mes = df_mes.filter(col("value").cast(DoubleType()) >= 0.0)
    df_mes = df_mes.filter(
        ~(
            ((col("parameter") == "pm25") & (col("value") > 1000.0)) |
            ((col("parameter") == "pm10") & (col("value") > 1000.0)) |
            ((col("parameter") == "no2")  & (col("value") > 1000.0)) |
            ((col("parameter") == "o3")   & (col("value") > 1000.0)) |
            ((col("parameter") == "so2")  & (col("value") > 1000.0)) |
            ((col("parameter") == "co")   & (col("value") > 10000.0))
        )
    )
    df_mes = df_mes.withColumn("datetime", to_timestamp(col("datetime")))
    df_mes = df_mes \
        .withColumn("hora",          hour(col("datetime"))) \
        .withColumn("dia_semana",    dayofweek(col("datetime"))) \
        .withColumn("mes",           month(col("datetime"))) \
        .withColumn("es_fin_semana", dayofweek(col("datetime")).isin([1, 7]))
    # --- FIN TRANSFORMACIONES ---

    count = df_mes.count()
    print(f'Registros leídos: {count:,}')

    df_mes.write.mode('overwrite').parquet(output_path)
    print(f'Guardado en: {output_path}')

    fin = datetime.now()
    duracion = fin - inicio
    minutos = int(duracion.total_seconds() // 60)
    segundos = int(duracion.total_seconds() % 60)

    print(f'Tiempo mes {mes}: {minutos}m {segundos}s')
    print(f'Mes {mes} completo — fin: {fin.strftime("%H:%M:%S")}')

print('\nETL de measurements completo')


Procesando mes: 01 — inicio: 12:40:16


26/04/06 12:40:18 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Registros leídos: 3,167,997


Guardado en: /app/data/measurements_2023_01.parquet
Tiempo mes 01: 53m 31s
Mes 01 completo — fin: 13:33:47

Procesando mes: 02 — inicio: 13:33:47


Registros leídos: 1,682,574


...

Guardado en: /app/data/measurements_2023_10.parquet
Tiempo mes 10: 64m 14s
Mes 10 completo — fin: 21:52:28

Procesando mes: 11 — inicio: 21:52:28


Registros leídos: 6,948,350


Guardado en: /app/data/measurements_2023_11.parquet
Tiempo mes 11: 68m 42s
Mes 11 completo — fin: 23:01:11

Procesando mes: 12 — inicio: 23:01:11


Registros leídos: 7,117,651


Guardado en: /app/data/measurements_2023_12.parquet
Tiempo mes 12: 70m 55s
Mes 12 completo — fin: 00:12:06

ETL de measurements completo


## 6. Verificación final

Leemos todos los parquets de mediciones y verificamos el total de registros.

In [10]:
# Leer todos los meses de una vez (mismo esquema -> Spark los une automáticamente)
df_measurements = spark.read.parquet(f'{output_dir}/measurements_2023_*.parquet')

print(f'Total registros 2023: {df_measurements.count():,}')
print('\nEsquema:')
df_measurements.printSchema()
print('\nMuestra:')
df_measurements.show(5)

Total registros 2023: 59,393,161

Esquema:
root
 |-- location_id: integer (nullable = true)
 |-- sensors_id: integer (nullable = true)
 |-- location: string (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- parameter: string (nullable = true)
 |-- units: string (nullable = true)
 |-- value: double (nullable = true)
 |-- hora: integer (nullable = true)
 |-- dia_semana: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- es_fin_semana: boolean (nullable = true)


Muestra:
+-----------+----------+--------------+-------------------+----------------+-------------------+---------+-----+-----+----+----------+---+-------------+
|location_id|sensors_id|      location|           datetime|             lat|                lon|parameter|units|value|hora|dia_semana|mes|es_fin_semana|
+-----------+----------+--------------+-------------------+----------------+-------------------+---------+-----+-----+

In [6]:
print("juntando todos los meses en un solo DataFrame...")
df_measurements = spark.read.parquet("/app/data/measurements_2023_*.parquet")


juntando todos los meses en un solo DataFrame...


In [7]:

print(f"Total de filas: {df_measurements.count()}")
print(f"Columnas: {df_measurements.columns}")
df_measurements.printSchema()
df_measurements.show(5)

Total de filas: 59393161
Columnas: ['location_id', 'sensors_id', 'location', 'datetime', 'lat', 'lon', 'parameter', 'units', 'value', 'hora', 'dia_semana', 'mes', 'es_fin_semana']
root
 |-- location_id: integer (nullable = true)
 |-- sensors_id: integer (nullable = true)
 |-- location: string (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- parameter: string (nullable = true)
 |-- units: string (nullable = true)
 |-- value: double (nullable = true)
 |-- hora: integer (nullable = true)
 |-- dia_semana: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- es_fin_semana: boolean (nullable = true)

+-----------+----------+--------------+-------------------+----------------+-------------------+---------+-----+-----+----+----------+---+-------------+
|location_id|sensors_id|      location|           datetime|             lat|                lon|parameter|units|value|hora|dia_semana|mes|es_fin

In [8]:
df_locations = spark.read.parquet("/app/data/Locations.parquet")
df_sensores = spark.read.parquet("/app/data/Sensores.parquet")
df_paises = spark.read.parquet("/app/data/Paises.parquet")

print("=== Locations ===")
df_locations.printSchema()

print("=== Sensores ===")
df_sensores.printSchema()

print("=== Paises ===")
df_paises.printSchema()

=== Locations ===
root
 |-- location_id: long (nullable = true)
 |-- location_name: string (nullable = true)
 |-- country_id: long (nullable = true)
 |-- country_code: string (nullable = true)
 |-- provider_id: long (nullable = true)
 |-- provider_name: string (nullable = true)
 |-- is_monitor: boolean (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- datetime_first: string (nullable = true)
 |-- datetime_last: string (nullable = true)

=== Sensores ===
root
 |-- sensor_id: long (nullable = true)
 |-- location_id: long (nullable = true)
 |-- param: string (nullable = true)
 |-- units: string (nullable = true)

=== Paises ===
root
 |-- ID_Paises: long (nullable = true)
 |-- Pais_Code: string (nullable = true)
 |-- Pais: string (nullable = true)



In [9]:
df_locations = df_locations.withColumn("location_id", col("location_id").cast("integer"))

In [ ]:
df_enriched = (
    df_measurements
    # Paso 1: trae country_code desde Locations
    .join(
        df_locations.select("location_id", "country_code", "provider_name", "is_monitor"),
        on="location_id",
        how="left"
    )
    # Paso 2: ahora df ya tiene country_code, entonces puede unirse con Paises
    .join(
        df_paises.select(col("Pais_Code").alias("country_code"), col("Pais").alias("country_name")),
        on="country_code",
        how="left"
    )
)

df_enriched.printSchema()
df_enriched.show(5)

In [ ]:
df_enriched.write.parquet("/app/data/measurements_enriched.parquet", mode="overwrite")

In [ ]:
df = spark.read.parquet("/app/data/measurements_enriched.parquet")
df.show(5)